## Get data directly in Python

In [1]:
import yfinance as yf
import pandas as pd
import time

# Your 10 diverse stocks across sectors
stocks = {
    "AAPL":  "Technology",
    "MSFT":  "Technology", 
    "JPM":   "Finance",
    "JNJ":   "Healthcare",
    "XOM":   "Energy",
    "AMZN":  "Consumer",
    "TSLA":  "Automotive",
    "GS":    "Finance",
    "PFE":   "Pharma",
    "SPY":   "Index (S&P 500)"  # the major index
}

all_data = {}

for ticker in stocks:
    try:
        df = yf.download(ticker, start="2019-01-01", end="2024-01-01", auto_adjust=True)
        all_data[ticker] = df
        print(f"✓ {ticker} downloaded — {len(df)} rows")
        time.sleep(1)  # avoid rate limit
    except Exception as e:
        print(f"✗ Error: {ticker} — {e}")

[*********************100%***********************]  1 of 1 completed


✓ AAPL downloaded — 1258 rows


[*********************100%***********************]  1 of 1 completed


✓ MSFT downloaded — 1258 rows


[*********************100%***********************]  1 of 1 completed


...

✓ GS downloaded — 1258 rows


[*********************100%***********************]  1 of 1 completed


✓ PFE downloaded — 1258 rows


[*********************100%***********************]  1 of 1 completed


✓ SPY downloaded — 1258 rows


## Data cleaning

In [2]:
# Combine all closing prices into one clean table
prices = pd.DataFrame({
    ticker: data["Close"].squeeze()
    for ticker, data in all_data.items()
})

prices.dropna(inplace=True)               # remove any missing days
prices.sort_index(inplace=True)           # sort by date
prices.index = pd.to_datetime(prices.index)  # ensure date format

print(prices.shape)   # rows = trading days, columns = 10 stocks
print(prices.head())

(1258, 10)
                 AAPL       MSFT        JPM         JNJ        XOM       AMZN  \
Date                                                                            
2019-01-02  37.503723  94.397163  80.836510  104.387878  50.001831  76.956497   
2019-01-03  33.768070  90.924446  79.687683  102.729126  49.234123  75.014000   
2019-01-04  35.209606  95.153290  82.625404  104.453239  51.049377  78.769501   
2019-01-07  35.131248  95.274651  82.682861  103.783226  51.314838  81.475502   
2019-01-08  35.800961  95.965454  82.526917  106.193741  51.687935  82.829002   

                 TSLA          GS        PFE         SPY  
Date                                                      
2019-01-02  20.674667  145.391663  28.818651  224.382553  
2019-01-03  20.024000  143.261871  28.012400  219.028137  
2019-01-04  21.179333  147.944016  28.652077  226.364624  
2019-01-07  22.330667  148.763840  28.805330  228.149460  
2019-01-08  22.356667  148.214478  28.938601  230.293015  


## data quality check

In [3]:
print("Missing values:\n", prices.isnull().sum())
print("\nDate range:", prices.index[0], "→", prices.index[-1])
print("\nStocks:", list(prices.columns))

Missing values:
 AAPL    0
MSFT    0
JPM     0
JNJ     0
XOM     0
AMZN    0
TSLA    0
GS      0
PFE     0
SPY     0
dtype: int64

Date range: 2019-01-02 00:00:00 → 2023-12-29 00:00:00

Stocks: ['AAPL', 'MSFT', 'JPM', 'JNJ', 'XOM', 'AMZN', 'TSLA', 'GS', 'PFE', 'SPY']


## Install libraries

In [4]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import time
import warnings
warnings.filterwarnings("ignore")
print("All imports successful")

All imports successful


## Fetch stock data

In [5]:
stocks = ["AAPL", "MSFT", "JPM", "JNJ", "XOM",
          "AMZN", "TSLA", "GS", "PFE", "SPY"]

all_data = {}
for ticker in stocks:
    try:
        df = yf.download(ticker, start="2019-01-01",
                         end="2024-01-01", auto_adjust=True, progress=False)
        all_data[ticker] = df
        print(f"✓ {ticker} fetched — {len(df)} trading days")
        time.sleep(0.5)
    except Exception as e:
        print(f"✗ {ticker} failed: {e}")

# Combine all closing prices
prices = pd.DataFrame({
    ticker: data["Close"].squeeze()
    for ticker, data in all_data.items()
})
prices.dropna(inplace=True)
print(f"\nClean price table: {prices.shape[0]} days × {prices.shape[1]} stocks")


✓ AAPL fetched — 1258 trading days
✓ MSFT fetched — 1258 trading days
✓ JPM fetched — 1258 trading days
✓ JNJ fetched — 1258 trading days
✓ XOM fetched — 1258 trading days
✓ AMZN fetched — 1258 trading days
✓ TSLA fetched — 1258 trading days
✓ GS fetched — 1258 trading days
✓ PFE fetched — 1258 trading days
✓ SPY fetched — 1258 trading days

Clean price table: 1258 days × 10 stocks


In [6]:
print(f"\nFinal shape: {prices.shape[0]} days × {prices.shape[1]} stocks")
prices.tail(5)   # preview last 5 rows


Final shape: 1258 days × 10 stocks


,AAPL,MSFT,JPM,JNJ,XOM,AMZN,TSLA,GS,PFE,SPY
Date,,,,,,,,,,
2023-12-22,191.609467,368.236603,158.576019,145.440445,94.430161,153.419998,252.539993,362.307556,24.148331,460.964417
2023-12-26,191.065125,368.315216,159.513855,146.076599,94.643265,153.410004,256.609985,363.221252,24.156830,462.910889
2023-12-27,191.164093,367.735291,160.470596,146.273102,94.198502,153.339996,261.440002,365.952942,24.326893,463.747833
2023-12-28,191.589676,368.924744,161.323181,146.488266,92.836395,153.380005,253.179993,367.789948,24.479944,463.923035
2023-12-29,190.550461,369.671906,161.133713,146.637939,92.641808,151.940002,248.479996,367.180756,24.479944,462.579956


## Daily Log Returns

In [7]:
# Formula: log_return = ln(Price_today / Price_yesterday)
# Why log returns? They are additive over time and normally distributed

log_returns = np.log(prices / prices.shift(1)).dropna()

print("\n--- Daily Log Returns Summary ---")
print(f"Shape: {log_returns.shape}  (rows=days, cols=stocks)")
print(log_returns.describe().round(4))  # mean, std, min, max per stock



--- Daily Log Returns Summary ---
Shape: (1257, 10)  (rows=days, cols=stocks)
            AAPL       MSFT        JPM        JNJ        XOM       AMZN  \
count  1257.0000  1257.0000  1257.0000  1257.0000  1257.0000  1257.0000   
mean      0.0013     0.0011     0.0005     0.0003     0.0005     0.0005   
std       0.0203     0.0192     0.0201     0.0125     0.0216     0.0222   
min      -0.1377    -0.1595    -0.1621    -0.0758    -0.1304    -0.1514   
25%      -0.0082    -0.0083    -0.0084    -0.0056    -0.0109    -0.0109   
50%       0.0014     0.0012     0.0007     0.0003     0.0005     0.0010   
75%       0.0124     0.0111     0.0096     0.0062     0.0115     0.0120   
max       0.1132     0.1329     0.1656     0.0769     0.1194     0.1269   

            TSLA         GS        PFE        SPY  
count  1257.0000  1257.0000  1257.0000  1257.0000  
mean      0.0020     0.0007    -0.0001     0.0006  
std       0.0408     0.0206     0.0169     0.0133  
min      -0.2365    -0.1359    -0.080

##  Portfolio Setup (Equal Weights)

In [8]:
n_stocks = len(stocks)
weights = np.array([1 / n_stocks] * n_stocks)   # equal weight = 10% each

# Portfolio daily return = weighted sum of individual returns
portfolio_daily_returns = log_returns @ weights  # matrix multiply

# Key stats
mean_return = portfolio_daily_returns.mean()     # avg daily return
std_return  = portfolio_daily_returns.std()      # daily volatility

print(f"\nPortfolio mean daily return : {mean_return:.6f}")
print(f"Portfolio daily std dev     : {std_return:.6f}")
print(f"Annualised return (approx)  : {mean_return * 252:.4f} ({mean_return * 252 * 100:.2f}%)")
print(f"Annualised volatility       : {std_return * np.sqrt(252):.4f} ({std_return * np.sqrt(252) * 100:.2f}%)")



Portfolio mean daily return : 0.000739
Portfolio daily std dev     : 0.014463
Annualised return (approx)  : 0.1863 (18.63%)
Annualised volatility       : 0.2296 (22.96%)


## Monte Carlo Simulation

In [9]:
# Idea: Simulate 252 trading days (1 year) × 10,000 times
# Each path = random daily returns drawn from historical distribution

N_SIMULATIONS  = 10_000   # number of random futures
N_DAYS         = 252      # 1 trading year
INITIAL_VALUE  = 100_000  # starting portfolio value ($100,000)
CONFIDENCE     = 0.95     # for VaR calculation

np.random.seed(42)        # reproducible results

print(f"\nRunning {N_SIMULATIONS:,} Monte Carlo simulations...")

# Generate all random returns at once (fast NumPy — no Python loop needed)
# Shape: (252 days, 10000 simulations)
random_returns = np.random.normal(
    loc   = mean_return,   # centre = historical mean
    scale = std_return,    # spread = historical volatility
    size  = (N_DAYS, N_SIMULATIONS)
)

# Cumulative return paths: compound daily returns day by day
# (1 + r1) * (1 + r2) * ... = cumprod along axis=0
cumulative_returns = np.cumprod(1 + random_returns, axis=0)

# Portfolio value at each day for each simulation
portfolio_paths = INITIAL_VALUE * cumulative_returns   # shape (252, 10000)

# Final portfolio value at end of 1 year (last row)
final_values = portfolio_paths[-1, :]                  # shape (10000,)

print(f"Simulation complete!")
print(f"\n--- Monte Carlo Results (1-Year Forecast) ---")
print(f"Starting value       : ${INITIAL_VALUE:>12,.2f}")
print(f"Mean ending value    : ${np.mean(final_values):>12,.2f}")
print(f"Median ending value  : ${np.median(final_values):>12,.2f}")
print(f"Best case  (99th %)  : ${np.percentile(final_values, 99):>12,.2f}")
print(f"Worst case (1st %)   : ${np.percentile(final_values, 1):>12,.2f}")

# Value at Risk (VaR) — how much could we LOSE at 95% confidence?
VaR_95 = INITIAL_VALUE - np.percentile(final_values, 5)
print(f"\nValue at Risk (95%)  : ${VaR_95:>12,.2f}")
print(f"  → There is 5% chance of losing more than ${VaR_95:,.2f} in 1 year")



Running 10,000 Monte Carlo simulations...
Simulation complete!

--- Monte Carlo Results (1-Year Forecast) ---
Starting value       : $  100,000.00
Mean ending value    : $  120,239.44
Median ending value  : $  117,276.04
Best case  (99th %)  : $  197,912.85
Worst case (1st %)   : $   68,361.68

Value at Risk (95%)  : $   19,547.28
  → There is 5% chance of losing more than $19,547.28 in 1 year


## Statistical Validation
    (Skewness, Kurtosis, Normality Test)

In [10]:
sim_skewness = stats.skew(final_values)
sim_kurtosis = stats.kurtosis(final_values)   # excess kurtosis (normal=0)
hist_skewness = stats.skew(portfolio_daily_returns)
hist_kurtosis = stats.kurtosis(portfolio_daily_returns)

# Shapiro-Wilk test: are returns normally distributed?
# Use sample of 1000 (Shapiro needs < 5000 samples)
_, p_value = stats.shapiro(np.random.choice(final_values, 1000))

print("\n--- Statistical Validation ---")
print(f"{'Metric':<30} {'Simulation':>14} {'Historical':>14}")
print("-" * 60)
print(f"{'Skewness':<30} {sim_skewness:>14.4f} {hist_skewness:>14.4f}")
print(f"{'Kurtosis (excess)':<30} {sim_kurtosis:>14.4f} {hist_kurtosis:>14.4f}")
print(f"\nNormality test (Shapiro-Wilk):")
print(f"  p-value = {p_value:.4f}")
if p_value > 0.05:
    print("  → Distribution is approximately normal ")
else:
    print("  → Distribution deviates from normal (fat tails present)")

# Interpretation guide
print("\n--- Skewness Interpretation ---")
if sim_skewness > 0:
    print("  Positive skew → more upside outcomes than downside (good)")
elif sim_skewness < -0.5:
    print("  Negative skew → tail risk on downside (caution)")
else:
    print("  Near-zero skew → roughly symmetric distribution")

print("\n--- Kurtosis Interpretation ---")
if sim_kurtosis > 1:
    print("  High kurtosis → fat tails, extreme events more likely")
else:
    print("  Low kurtosis → thin tails, fewer extreme surprises")


--- Statistical Validation ---
Metric                             Simulation     Historical
------------------------------------------------------------
Skewness                               0.6598        -0.8218
Kurtosis (excess)                      0.6906        10.1034

Normality test (Shapiro-Wilk):
  p-value = 0.0000
  → Distribution deviates from normal (fat tails present)

--- Skewness Interpretation ---
  Positive skew → more upside outcomes than downside (good)

--- Kurtosis Interpretation ---
  Low kurtosis → thin tails, fewer extreme surprises
